In [1]:
%pylab inline
%config InlineBackend.figure_format = 'retina'
from ipywidgets import interact
import scipy.sparse as sparse
from scipy.sparse import linalg as spla
from scipy import interpolate as interp

Populating the interactive namespace from numpy and matplotlib


# Model problem
The advection equation
$$ \frac{\partial u}{\partial t} + a \frac{\partial u}{\partial x} = 0 , \quad x \in [0, L],$$
with periodic initial data
$$ u(x, 0) = \eta(x), \quad  \eta(0) = \eta(L), \quad \eta'(0) = \eta'(L)$$
The exact solution is
$$u(x, t) = \eta(x - at \mod L). $$
We will consider the example $L=25$ with
$$ \eta(x) = e^{-20(x-2)^2} + e^{-(x-5)^2}, $$
which is not exactly periodic, but it is close.


We will discretize space with $m+1$ intervals and $m+2$ grid points $x_j = jh$, $j=0, 1, \ldots, m+1$ and $h=L/(m+1)$. One minor difference compared to our elliptic and parabolic examples is that there is only one boundary condition $u(0, t) = u(L, t)$ which we impose at grid point $x_0 = 0$. The schemes will be written for linear systems with $m+1$ unknowns (compared to $m$ unknowns in previous weeks). The center difference is used in many of the following methods. The corresponding $(m+1)\times (m+1)$ matrix with periodic boundary conditions is
$$ A = - \frac{a}{2h}
  \begin{bmatrix}
    0& 1 & & & & -1\\
    -1& 0 & 1& & & \\
    &  -1& 0& 1& & \\
    &  & \ddots & \ddots & \ddots & \\
    &  & & -1& 0& 1\\
   1 &  & & & -1 & 0
  \end{bmatrix}
$$

We will discretize the time interval $[0, T]$ with $t_n = kn$, $n=0, 1, \ldots, N$, where $k = T/N$.

In [20]:
def eta1(x): ## approximately periodic
    return exp(-20*(x-2)**2) + exp(-(x-5)**2)

def eta2(x): ## approximately periodic
    xi0 = 25.
    beta = 0.1
    return sin(xi0*(2*pi/L)*x)*exp(-beta*(x-L/2)**2)

def eta3(x):
    umax = 1.1
    return umax*((x < 0.7*L)&(x > 0.3*L))

#### Select initial data ####
# eta = eta1
# eta = eta2
eta = eta3

##########################
def uexact(x, t, a):
    y = (x - a*t) % L
    return eta(y)

# Forward Euler (for the advection equation)
$$ U^{n+1} = [I + kA]U^n $$

In [26]:
## parameters
a = 1.2

## time discretization
N_time_steps = 100
T = 5. # max time
k = T/N_time_steps # time step
t = linspace(0, T, N_time_steps+1)

## space discretization
L = 25.
m = 200
h = L/(m+1)
x = linspace(0, L, m+2)

nu = a*k/h
print('nu=', nu)
print('k=', around(k, 4), 'h^2=', around(h**2, 4))

e = ones(m+1)
A = -a/(2*h)*sparse.spdiags(
    [e, -e, e, -e], [-m, -1, 1, m], m+1, m+1)
I = sparse.eye(m+1)
B = I + k*A
# print(around(A.toarray(), 2))

utrue = uexact(x[None, :], t[:, None], a)
u = utrue.copy()

for n in arange(N_time_steps):
    u[n+1, 1:] = B@u[n, 1:]

u[:, 0] = u[:, -1]

## visualize solution
@interact(n=(0, N_time_steps, 1))
def plotfn(n=0):
    fig = figure(1, [16, 4])
    fig.add_subplot(121)
    plot(x, utrue[n], 'k')
    plot(x, u[n], '-o', ms=4)
    #ylim(0, 1)
    xlabel(r'$x$', fontsize=24)
    ylabel(r'$u$', fontsize=24)

    fig.add_subplot(122)
    plot(x, u[n] - utrue[n], 'k')
    xlabel(r'$x$', fontsize=24)
    ylabel(r'error', fontsize=24);
    show();

nu= 0.4824
k= 0.05 h^2= 0.0155


interactive(children=(IntSlider(value=0, description='n'), Output()), _dom_classes=('widget-interact',))

# Leap-Frog
$$ U^{n+1} = U^{n-1} + 2kAU^n $$

In [27]:
## parameters
a = 1.2

## time discretization
N_time_steps = 200
T = 10. # max time
k = T/N_time_steps # time step
t = linspace(0, T, N_time_steps+1)

## space discretization
L = 25.
m = 200
h = L/(m+1)
x = linspace(0, L, m+2)

nu = a*k/h
print('nu=', nu)

e = ones(m+1)
A = -a/(2*h)*sparse.spdiags(
    [e, -e, e, -e], [-m, -1, 1, m], m+1, m+1)
I = sparse.eye(m+1)

utrue = uexact(x[None, :], t[:, None], a)
u = utrue.copy()


for n in arange(1, N_time_steps):
    u[n+1, 1:] = u[n-1, 1:] + 2*k*A@u[n, 1:]

u[:, 0] = u[:, -1]

## visualize solution
@interact(n=(0, N_time_steps, 1))
def plotfn(n=0):
    fig = figure(1, [16, 4])
    fig.add_subplot(121)
    plot(x, utrue[n], 'k')
    plot(x, u[n], '-o', ms=4)
    #ylim(0, 1)
    xlabel(r'$x$', fontsize=24)
    ylabel(r'$u$', fontsize=24)

    fig.add_subplot(122)
    plot(x, u[n] - utrue[n], 'k')
    xlabel(r'$x$', fontsize=24)
    ylabel(r'error', fontsize=24);
    show();

nu= 0.4824


interactive(children=(IntSlider(value=0, description='n', max=200), Output()), _dom_classes=('widget-interact'…

# Lax-Friedrichs
$$ U^{n+1} = \left[I + kA + k\left(\frac{h^2}{2k}\right)A_{\rm 2nd}\right]U^n,$$
where the centered second derivative matrix (with periodic boundaries) is given by
$$ A_{\rm 2nd} =  \frac{1}{h^2}
  \begin{bmatrix}
    -2& 1 & & & & 1\\
    1& -2 & 1& & & \\
    &  1& -2& 1& & \\
    &  & \ddots & \ddots & \ddots & \\
    &  & & 1& -2& 1\\
   1 &  & & & 1 & -2
  \end{bmatrix}
$$

In [29]:
## parameters
a = 1.2

## time discretization
N_time_steps = 200
T = 10. # max time

## This choice of T will yield a*k/h = +/- 1
# T = L*N_time_steps/(absolute(a)*(m+1))
# print('T=', T)

k = T/N_time_steps # time step
t = linspace(0, T, N_time_steps+1)

## space discretization
L = 25.
m = 200
h = L/(m+1)
x = linspace(0, L, m+2)

nu = a*k/h
print('nu=', nu)

e = ones(m+1)
A = -a/(2*h)*sparse.spdiags(
    [e, -e, e, -e], [-m, -1, 1, m], m+1, m+1)
epsilon = h**2/(2*k)
A_2nd = epsilon/h**2*sparse.spdiags(
    [e, e, -2*e, e, e], [-m, -1, 0, 1, m], m+1, m+1)
I = sparse.eye(m+1)
B = I + k*A + k*A_2nd

utrue = uexact(x[None, :], t[:, None], a)
u = utrue.copy()

for n in arange(N_time_steps):
    u[n+1, 1:] = B@u[n, 1:]

u[:, 0] = u[:, -1]

## visualize solution
@interact(n=(0, N_time_steps, 1))
def plotfn(n=0):
    fig = figure(1, [16, 4])
    fig.add_subplot(121)
    plot(x, utrue[n], 'k')
    plot(x, u[n], '-o', ms=4)
    #ylim(0, 1)
    xlabel(r'$x$', fontsize=24)
    ylabel(r'$u$', fontsize=24)

    fig.add_subplot(122)
    plot(x, u[n] - utrue[n], 'k')
    xlabel(r'$x$', fontsize=24)
    ylabel(r'error', fontsize=24);
    show();

nu= 0.4824


interactive(children=(IntSlider(value=0, description='n', max=200), Output()), _dom_classes=('widget-interact'…

# Lax-Wendroff
$$ U^{n+1} = \left[I + kA + k\left(\frac{a^2k}{2}\right)A_{\rm 2nd}\right]U^n,$$
where the centered second derivative matrix (with periodic boundaries) is given by
$$ A_{\rm 2nd} =  \frac{1}{h^2}
  \begin{bmatrix}
    -2& 1 & & & & 1\\
    1& -2 & 1& & & \\
    &  1& -2& 1& & \\
    &  & \ddots & \ddots & \ddots & \\
    &  & & 1& -2& 1\\
   1 &  & & & 1 & -2
  \end{bmatrix}
$$

In [30]:
## parameters
a = 1.2

## time discretization
N_time_steps = 200
T = 10. # max time
## This choice of T will yield a*k/h = +/- 1
# T = L*N_time_steps/(absolute(a)*(m+1))
# print('T=', T)
k = T/N_time_steps # time step
t = linspace(0, T, N_time_steps+1)

## space discretization
L = 25.
m = 200
h = L/(m+1)
x = linspace(0, L, m+2)

nu = a*k/h
print('nu=', nu)

e = ones(m+1)
A = -a/(2*h)*sparse.spdiags([e, -e, e, -e], [-m, -1, 1, m], m+1, m+1)
epsilon = a**2*k/2.
A_2nd = epsilon/h**2*sparse.spdiags([e, e, -2*e, e, e], [-m, -1, 0, 1, m], m+1, m+1)
I = sparse.eye(m+1)
B = I + k*A + k*A_2nd

utrue = uexact(x[None, :], t[:, None], a)
u = utrue.copy()

for n in arange(N_time_steps):
    u[n+1, 1:] = B@u[n, 1:]

u[:, 0] = u[:, -1]

## visualize solution
@interact(n=(0, N_time_steps, 1))
def plotfn(n=0):
    fig = figure(1, [16, 4])
    fig.add_subplot(121)
    plot(x, utrue[n], 'k')
    plot(x, u[n], '-o', ms=4)
    #ylim(0, 1)
    xlabel(r'$x$', fontsize=24)
    ylabel(r'$u$', fontsize=24)

    fig.add_subplot(122)
    plot(x, u[n] - utrue[n], 'k')
    xlabel(r'$x$', fontsize=24)
    ylabel(r'error', fontsize=24);
    show();

nu= 0.4824


interactive(children=(IntSlider(value=0, description='n', max=200), Output()), _dom_classes=('widget-interact'…

# Upwind

  - For $a>0$
$$ U^{n+1} = \left[I + k A_{(>)} \right]U^n,$$
  - For $a< 0$
$$ U^{n+1} = \left[I + k A_{(<)} \right]U^n,$$
where
$$ A_{(>)} =  -\frac{a}{h}
  \begin{bmatrix}
    1& & & & & -1\\
    -1& 1 & & & & \\
    &  -1& 1& & & \\
    &  & \ddots & \ddots &  & \\
    &  & & -1& 1& \\
    &  & & & -1 & 1
  \end{bmatrix}
$$
and $A_{(<)} = -A_{(>)}^T$.


In [25]:
## parameters
a = 1.2

## time discretization
N_time_steps = 400
T = 20. # max time
## This choice of T will yield a*k/h = +/- 1
# T = L*N_time_steps/(absolute(a)*(m+1))
# print('T=', T)
k = T/N_time_steps # time step
t = linspace(0, T, N_time_steps+1)

## space discretization
L = 25.
m = 200
h = L/(m+1)
x = linspace(0, L, m+2)

nu = a*k/h
print('nu=', nu)

e = ones(m+1)
A1 = -a/h*sparse.spdiags(
    [e, -e, e], [-m, 0, 1], m+1, m+1)
A2 = -A1.T
I = sparse.eye(m+1)
if a < 0:
    B = I + k*A1
else:
    B = I + k*A2

utrue = uexact(x[None, :], t[:, None], a)
u = utrue.copy()

for n in arange(N_time_steps):
    u[n+1, 1:] = B@u[n, 1:]

u[:, 0] = u[:, -1]

## visualize solution
@interact(n=(0, N_time_steps, 1))
def plotfn(n=0):
    fig = figure(1, [16, 4])
    fig.add_subplot(121)
    plot(x, utrue[n], 'k')
    plot(x, u[n], '-o', ms=4)
    #ylim(0, 1)
    xlabel(r'$x$', fontsize=24)
    ylabel(r'$u$', fontsize=24)

    fig.add_subplot(122)
    plot(x, u[n] - utrue[n], 'k')
    xlabel(r'$x$', fontsize=24)
    ylabel(r'error', fontsize=24);
    show();

nu= 0.4824


interactive(children=(IntSlider(value=0, description='n', max=400), Output()), _dom_classes=('widget-interact'…